# Make a PDF invoice with Fullbleed

Edit Python data, render it with HTML/CSS, inspect the PDF, and download it.
The example uses fictional data and the font bundled with Fullbleed.

**In Colab:** choose **Runtime > Run all**. Sign in if prompted and use the
standard CPU runtime; this example needs no GPU or paid plan. To customize the
invoice, edit step 2 and run steps 2–5 again.

**In Jupyter:** run the cells in order with a Python 3.10–3.14 kernel.

[Documentation](https://docs.fullbleed.dev/) ·
[Source and MIT license](https://github.com/fullbleed-engine/fullbleed-official) ·
[Ask a question](https://github.com/fullbleed-engine/fullbleed-official/discussions)

Colab runs your code on a hosted machine. Files in the runtime are temporary;
download the PDF before leaving. See [Colab's FAQ](https://research.google.com/colaboratory/faq.html)
for its free tier and runtime limits.

## 1. Install the example's tested release

The wheel includes the PDF engine, CLI, and fonts. Jupyter supplies the notebook
display tools; they are not dependencies of the Fullbleed package.

In [ ]:
%pip install --quiet "fullbleed==2.4.0"

## 2. Edit the invoice data

Change the customer, descriptions, quantities, or unit prices. Write money as
strings so `Decimal` can calculate the total without binary floating-point
rounding. This small example omits tax and discounts.

In [ ]:
invoice = {
    "number": "INV-1042",
    "issued": "2026-10-01",
    "due": "2026-10-31",
    "business": "Northstar Studio",
    "customer": "Maple & Finch",
    "items": [
        {"description": "Design workshop", "quantity": 8, "unit_price": "125.00"},
        {"description": "Implementation", "quantity": 6, "unit_price": "95.00"},
        {"description": "Review and handoff", "quantity": 2, "unit_price": "150.00"},
    ],
}

## 3. Turn the data into HTML and CSS

Text values are escaped before entering HTML. Change the teal color, page
margins, or table styles to make this your own.

In [ ]:
from decimal import Decimal
from html import escape


def money(value):
    return f"USD {value:,.2f}"


amounts = [Decimal(item["unit_price"]) * item["quantity"] for item in invoice["items"]]
total = sum(amounts, Decimal("0.00"))
rows = "".join(
    f"<tr><td>{escape(item['description'])}</td>"
    f"<td class='number'>{item['quantity']}</td>"
    f"<td class='number'>{money(Decimal(item['unit_price']))}</td>"
    f"<td class='number'>{money(amount)}</td></tr>"
    for item, amount in zip(invoice["items"], amounts)
)
html = f"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><title>Invoice</title></head>
<body><main>
  <p class="brand">{escape(invoice['business'])}</p>
  <h1>Invoice {escape(invoice['number'])}</h1>
  <p class="dates">Issued {escape(invoice['issued'])} · Due {escape(invoice['due'])}</p>
  <div class="recipient"><p class="label">BILL TO</p>
    <p>{escape(invoice['customer'])}</p></div>
  <table><thead><tr><th>Description</th><th class="number">Qty</th>
    <th class="number">Unit price</th><th class="number">Amount</th></tr></thead>
    <tbody>{rows}</tbody></table>
  <p class="total">Total due <strong>{money(total)}</strong></p>
  <p class="note">Thank you for your business.</p>
</main></body></html>"""
css = """
@page { size: A4; margin: 20mm; }
body { font-family: Inter, sans-serif; color: #18312e; font-size: 10pt;
       line-height: 1.45; margin: 0; }
.brand { color: #175c52; font-size: 13pt; font-weight: 700; margin: 0 0 24pt; }
h1 { font-size: 27pt; margin: 0 0 6pt; }
.dates, .label, .note { color: #526762; }
.dates { margin: 0 0 25pt; }
.recipient { margin-bottom: 24pt; }
.recipient p { margin: 0 0 4pt; }
.label { font-size: 8pt; font-weight: 700; }
table { width: 100%; border-collapse: collapse; }
th, td { text-align: left; padding: 10pt 8pt; border-bottom: 0.6pt solid #d6e2df; }
th { background: #edf4f1; color: #175c52; font-size: 9pt; }
.number { text-align: right; white-space: nowrap; }
.total { margin: 24pt 0 0; text-align: right; font-size: 14pt; }
.total strong { margin-left: 16pt; }
.note { margin-top: 42pt; font-size: 9pt; }
"""
print(f"{len(invoice['items'])} items · {money(total)}")

## 4. Render, inspect, and preview

Fullbleed embeds its bundled Inter font and saves the HTML, CSS, PDF, and PNG
preview under `output/`. The checks below confirm that the invoice number,
customer, and total appear in the PDF's extracted text. They also compare a
second render's bytes with the first. These checks do not certify a PDF standard.

In [ ]:
from hashlib import sha256
from importlib import metadata, resources
from pathlib import Path
import json
import fullbleed

output = Path("output")
output.mkdir(exist_ok=True)
pdf_path = output / "invoice.pdf"
font = resources.files("fullbleed_assets").joinpath("fonts/Inter-Variable.ttf")
engine = fullbleed.PdfEngine(
    font_files=[str(font)],
    document_title=f"Invoice {invoice['number']}",
    document_lang="en",
)
pdf = engine.render_pdf(html, css)
pdf_path.write_bytes(pdf)
(output / "invoice.html").write_text(html, encoding="utf-8")
(output / "invoice.css").write_text(css, encoding="utf-8")

inspection = dict(fullbleed.inspect_pdf(str(pdf_path)))
extracted = fullbleed.extract_pdf_page_texts(str(pdf_path))
text = "\n".join(page.get("text", "") for page in extracted["pages"])
markers = [invoice["number"], invoice["customer"], money(total)]
missing = [marker for marker in markers if marker not in text]
assert not missing, f"Missing PDF text: {missing}"
assert engine.render_pdf(html, css) == pdf, "Repeated render produced different bytes"
report = {
    "fullbleed_version": metadata.version("fullbleed"),
    "page_count": inspection["page_count"],
    "bytes": len(pdf),
    "sha256": sha256(pdf).hexdigest(),
    "text_checks": "passed",
    "repeat_render_identical": True,
}
(output / "verification.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))

In [ ]:
from IPython.display import Image, display

preview_paths = engine.render_finalized_pdf_image_pages_to_dir(
    str(pdf_path), str(output / "preview"), 110, "invoice"
)
print(f"Previewing page 1 of {len(preview_paths)}. Open the PDF to review every page.")
display(Image(filename=str(preview_paths[0]), width=600))

## 5. Download your PDF

In Colab, this cell starts a PDF download. You can also use the Files sidebar to
download the HTML, CSS, preview, and verification report from `output/`.
In local Jupyter, use the link below or open `output/invoice.pdf` directly.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(pdf_path))
else:
    from IPython.display import FileLink
    display(FileLink(str(pdf_path)))

## Keep building

- [Run locally](https://docs.fullbleed.dev/getting-started/quickstart/): the same
  Python API works in scripts and applications with `python -m pip install fullbleed`.
- [Invoice examples](https://docs.fullbleed.dev/guides/invoices/): JSON and CSV workflows.
- [Generate many documents](https://docs.fullbleed.dev/guides/bank-statements/):
  compiled fixed and reflowing variable-data examples.
- [Supported CSS](https://docs.fullbleed.dev/css-coverage/): check your template's layout needs.
- [Share your document or ask for help](https://github.com/fullbleed-engine/fullbleed-official/discussions).

This notebook is MIT licensed, like the
[Fullbleed repository](https://github.com/fullbleed-engine/fullbleed-official/blob/master/LICENSE).